# 04. Observable evidence: is the data consistent with low to moderate alpha?

Tests **H4** and **H5**. Supporting evidence for bounding alpha, not the
headline finding. The headline is notebook 01.

| Hypothesis | Claim | Kill condition |
|---|---|---|
| H4 | If alpha were high, food delivery order frequency should show pressure as Bistro scaled | Frequency contribution turns persistently negative while MTU stays positive |
| H5 | Category maturity, not Bistro, explains food delivery's growth path | Swiggy food delivery grows materially faster than Zomato through the scale up |

Kill conditions were written before the analysis was run. They are not adjusted
to fit the result.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt

from src import config, panel, metrics, viz, unit_economics as ue

viz.apply_style()
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

panel.assert_reviewed()  # refuses to run on an unvalidated panel

df = pd.read_csv(config.PROCESSED / "eternal_quarterly_enriched.csv")
df = metrics.sort_by_quarter(df)
quarters = df["quarter"].tolist()
len(df)

## The growth path, against the rollout

Plot food delivery NOV growth against Bistro kitchen count. This is context for
H4 rather than a test of it. Two lines moving in opposite directions is
consistent with cannibalisation and with a dozen other things, so read it as
framing and let the decomposition do the work.

In [ ]:
g = metrics.add_growth(df, ["fd_nov", "qc_nov"])

fig, ax1 = plt.subplots()
ax1.plot(quarters, g["fd_nov_yoy"], marker="o",
         color=viz.PALETTE["fd"], label="Food delivery NOV, YoY")
viz.pct_axis(ax1)
ax1.set_ylabel("Food delivery NOV growth, YoY")

ax2 = ax1.twinx()
ax2.bar(quarters, df["bistro_kitchens"], alpha=0.25,
        color=viz.PALETTE["qc"], label="Bistro kitchens")
ax2.set_ylabel("Bistro kitchens")
ax2.grid(False)

ax1.set_title("Food delivery growth vs Bistro rollout")
plt.xticks(rotation=45, ha="right")
fig.legend(loc="upper left", bbox_to_anchor=(0.12, 0.88))
viz.caption(fig, "Source: Eternal quarterly shareholder letters. "
                 "Kitchen counts from press reports, approximate.")
viz.save(fig, "h4_fd_growth_vs_bistro")
plt.show()

### Read this as framing, not as a verdict

A pre and post split on roughly ten quarters has no statistical power. Report
it as a descriptive contrast for a chart and never attach a p-value to it.

In [ ]:
summary = metrics.pre_post_summary(g, "fd_nov_yoy", config.BISTRO_START_QUARTER)
print(f"Mean FD NOV growth before vs after {config.BISTRO_START_QUARTER}:")
summary

**Caveat.** With roughly ten quarters there is no statistical power here.
This is a descriptive split for framing a chart. Do not attach a p-value to it.

## H4: decomposition into MTU, frequency and AOV

Cannibalisation does not make users delete the app. It makes them order less
often. So it should appear in the frequency term, with MTU still growing.
Frequency is the term alpha would move.

Decomposition is done in log space so the driver contributions sum exactly.

In [ ]:
dec = metrics.decompose_growth(df, segment="fd")
print(f"Max residual: {dec['residual'].abs().max():.2e} (should be ~0)")
dec[["quarter", "nov_yoy", "mtu_share", "frequency_share", "aov_share"]]

In [ ]:
d = dec.dropna(subset=["total_log_growth"])
fig, ax = plt.subplots()
ax.bar(d["quarter"], d["mtu_log_contrib"], label="MTU", color=viz.PALETTE["fd"])
ax.bar(d["quarter"], d["frequency_log_contrib"], bottom=d["mtu_log_contrib"],
       label="Frequency", color=viz.PALETTE["qc"])
ax.bar(d["quarter"], d["aov_log_contrib"],
       bottom=d["mtu_log_contrib"] + d["frequency_log_contrib"],
       label="AOV", color=viz.PALETTE["accent"])
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("H4: what drives food delivery NOV growth")
ax.set_ylabel("Log contribution to YoY growth")
ax.legend()
plt.xticks(rotation=45, ha="right")
viz.caption(fig, "Log-space decomposition of NOV = MTU x frequency x AOV. "
                 "Contributions sum to total log growth by construction.")
viz.save(fig, "h4_nov_decomposition")
plt.show()

### Verdict on H4

The frequency bar is the cannibalisation signal. Persistently negative
frequency contribution alongside positive MTU contribution is the pattern a
high alpha predicts. State whether it appears.

If frequency holds up through the scale up, that is evidence for a low alpha
and it tightens the range in notebook 01.

## H5: Swiggy as a control group

If Indian food delivery is simply maturing, Swiggy should decelerate in
parallel. Swiggy has no Bistro, so a shared slowdown points to category
maturity rather than internal substitution.

**Name the contamination.** Swiggy runs Bolt and Snacc, so it is not a clean
control. State the direction that biases the comparison: a contaminated control
that is also cannibalising itself makes the two series look more alike than
they should, which flatters the maturity story.

In [ ]:
sw = pd.read_csv(config.SWIGGY_PANEL)
spread = metrics.growth_spread(df, sw, "fd_nov", "swiggy_fd_gov")

fig, ax = plt.subplots()
ax.plot(spread["quarter"], spread["treated_yoy"], marker="o",
        color=viz.PALETTE["fd"], label="Zomato food delivery")
ax.plot(spread["quarter"], spread["control_yoy"], marker="s",
        color=viz.PALETTE["control"], label="Swiggy food delivery")
viz.pct_axis(ax)
ax.set_title("H5: Zomato vs Swiggy food delivery growth")
ax.set_ylabel("YoY growth")
ax.legend()
plt.xticks(rotation=45, ha="right")
viz.caption(fig, "Swiggy is an imperfect control: it operates Bolt and Snacc. "
                 "Zomato NOV vs Swiggy GOV are not identical measures. "
                 "Descriptive spread, not a causal estimate.")
viz.save(fig, "h5_swiggy_control")
plt.show()

spread

### Parallel trends

A difference-in-differences reading requires the two series to track together
before treatment. Inspect the pre-Bistro quarters. If they already diverge, the
spread cannot be attributed to Bistro and must be reported as descriptive only.


## Section verdict

Summarise H4 and H5 against their stated kill conditions. If a hypothesis is
disproved, say so plainly. A clean disproof is a stronger result than a
confirmation reached by adjusting the test.

Then carry one thing forward: what these two results imply for the plausible
range of alpha in notebook 01.